# Identify analyses to rerun

Build every requested country–analysis pair (`oxcgrt_included` × `get_analyses_for_country`), drop pairs already usable in `FULL_RUN`, and write the remainder to `data/config/rerun_pairs.json` for `scripts/massive/jtrauer/launch.sh`.

**Usable** (`analysis_output_status` returns `"usable"`): non-empty `updates.h5`, `spaghetti.h5`, and `idata_filtered.nc` ≥ `MIN_IDATA_BYTES`.

**Skipped**: mobility analyses logged as `{analysis} data not available`, or with no matching file in `data/mobility/`. These are omitted from the rerun list.

**Job order**: `FULL_RUN` in `constants.py` — first usable copy wins. Prepend a new job ID there after a remote run if its outputs should take precedence.

When `remaining` is 0, no targeted rerun is needed.


In [ ]:
import json

import pandas as pd

from emu_renewal.constants import ANALYSIS_TYPES, DATA_PATH, OUTPUTS_PATH, FULL_RUN
from emu_renewal.run import get_analyses_for_country
from emu_renewal.utils import analysis_output_status

In [ ]:
job_ids = FULL_RUN
countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))

In [ ]:
def classify_analysis(iso3, analysis, run_path, log_text):
    analysis_path = run_path / iso3 / analysis
    if log_text and f"{analysis} data not available" in log_text:
        return "skipped"
    return analysis_output_status(analysis_path)


def classify_job(run_id):
    run_path = OUTPUTS_PATH / run_id
    status = pd.DataFrame(index=countries, columns=ANALYSIS_TYPES)
    for iso3 in countries:
        log_path = run_path / iso3 / "run.log"
        log_text = log_path.read_text() if log_path.exists() else None
        requested_types = get_analyses_for_country(iso3)
        for analysis in ANALYSIS_TYPES:
            if analysis not in requested_types:
                status.loc[iso3, analysis] = "not requested"
            else:
                status.loc[iso3, analysis] = classify_analysis(iso3, analysis, run_path, log_text)
    return status


statuses = {job: classify_job(job) for job in job_ids}
pd.concat(
    {job: s.apply(pd.Series.value_counts).fillna(0).astype(int) for job, s in statuses.items()},
    axis=1,
).fillna(0).astype(int)

In [ ]:
MOB_SCALER_FILES = {
    "g_mob": "gmob_data.csv",
    "fb_visited_mob": "fbmob_data.csv",
    "fb_singletile_mob": "fbsingletile_data.csv",
}


def has_mobility_scaler(iso3, analysis):
    filename = MOB_SCALER_FILES.get(analysis)
    if filename is None:
        return True
    return (DATA_PATH / "mobility" / f"{iso3}_{filename}").exists()


def pairs_from_mask(mask):
    stacked = mask.stack()
    return list(stacked[stacked].index)


requested = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
for iso3 in countries:
    requested.loc[iso3, get_analyses_for_country(iso3)] = True

skipped = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
claimed = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
summary = {"requested": int(requested.to_numpy().sum())}
for job in job_ids:
    skipped = skipped | (statuses[job] == "skipped")
    usable = statuses[job] == "usable"
    summary[f"available from {job}"] = int((requested & usable & ~claimed).to_numpy().sum())
    claimed = claimed | usable

no_scaler = pd.DataFrame(
    [
        [has_mobility_scaler(iso3, analysis) for analysis in ANALYSIS_TYPES]
        for iso3 in countries
    ],
    index=countries,
    columns=ANALYSIS_TYPES,
)

summary["skipped (logged)"] = int((requested & skipped).to_numpy().sum())
summary["no mobility scaler CSV"] = int((requested & ~no_scaler).to_numpy().sum())
need = requested & ~skipped & ~claimed & no_scaler
rerun_pairs = sorted(pairs_from_mask(need))
summary["remaining"] = len(rerun_pairs)
pd.Series(summary)

In [ ]:
if rerun_pairs:
    remaining_df = pd.DataFrame(
        [
            {
                "iso3": iso3,
                "analysis": analysis,
                **{job: analysis_output_status(OUTPUTS_PATH / job / iso3 / analysis) for job in job_ids},
            }
            for iso3, analysis in rerun_pairs
        ]
    )
    display(remaining_df)
    display(pd.Series([analysis for _, analysis in rerun_pairs]).value_counts())
else:
    print("No reruns required")

In [ ]:
json.dump(rerun_pairs, open(DATA_PATH / "config/rerun_pairs.json", "w"))